# Self-Checking Diagnostic — KL Divergence over Rollouts

Tests whether KL divergence between rollouts is a meaningful correctness signal, and whether that signal lives in the **reasoning region** or only the **answer region** (memorization diagnostic).

**Runtime:** A100 recommended. T4 will work but is slow (~3–4x slower per sample).

**No training needed** — this runs on the base model only.

## 1. Clone Repository

In [ ]:
import os

# Clone the repo (skip if already cloned, e.g. after a kernel restart without runtime reset)
REPO_URL  = "https://github.com/ChristianDe-fabolous/Self-Distillation.git"
REPO_NAME = "Self-Distillation"

if not os.path.isdir(f"/content/{REPO_NAME}"):
    !git clone {REPO_URL} /content/{REPO_NAME}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C /content/{REPO_NAME} pull

os.chdir(f"/content/{REPO_NAME}")
print(f"Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

## 2. Install Dependencies

In [ ]:
# Install from the root requirements.txt.
# vllm and flashinfer are skipped — not needed for this script (uses transformers directly).
!pip install -q \
    torch==2.9.0 \
    transformers==4.57.1 \
    accelerate==1.11.0 \
    datasets==4.3.0 \
    peft==0.17.1 \
    numpy==2.2.6

## 3. HuggingFace Login (for gated models)

Qwen2.5-7B-Instruct is not gated, so this step can be skipped.
If you switch to a Llama model, you will need to authenticate.

In [ ]:
# Uncomment if needed
# from huggingface_hub import login
# login()

## 4. Verify GPU and Dataset

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from datasets import load_from_disk

dataset = load_from_disk("data/science_data/eval_data")
print(f"Science eval: {len(dataset)} examples")
print("Columns:", dataset.column_names)
print("\nExample prompt (first message):")
print(dataset[0]['prompt'][1]['content'][:300])
print("\nAnswer:", dataset[0]['answer'])

## 5. Configuration

Adjust these before running.

In [ ]:
MODEL_PATH     = "Qwen/Qwen2.5-7B-Instruct"  # or local path if already on Drive
N_ROLLOUTS     = 8       # rollouts per question — more = better signal, slower
TEMPERATURE    = 0.7     # must be > 0 for diverse rollouts
N_SAMPLES      = 50      # start small; set to 507 for full eval
MAX_NEW_TOKENS = 512
SEED           = 42
OUTPUT_DIR     = "experiments/results/self_checking"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Output will be saved to:", os.path.abspath(OUTPUT_DIR))

## 6. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading tokenizer: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print(f"Loading model: {MODEL_PATH}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()
device = next(model.parameters()).device
print(f"Model loaded on: {device}")

## 7. Run Self-Checking Diagnostic

In [ ]:
# Run the script directly — all output and logs go to samples.jsonl incrementally.
# Safe to interrupt; everything written so far is on disk.

!python experiments/self_checking.py \
    --model_path     {MODEL_PATH} \
    --n_rollouts     {N_ROLLOUTS} \
    --temperature    {TEMPERATURE} \
    --n_samples      {N_SAMPLES} \
    --max_new_tokens {MAX_NEW_TOKENS} \
    --seed           {SEED} \
    --output_dir     {OUTPUT_DIR}

## 8. Inspect Results

In [ ]:
import json

with open(f"{OUTPUT_DIR}/summary.json") as f:
    summary = json.load(f)

print("=" * 55)
print("SELECTION ACCURACY")
print(f"  random       : {summary['accuracy_random']:.4f}")
print(f"  majority     : {summary['accuracy_majority']:.4f}")
print(f"  kl_centroid  : {summary['accuracy_kl_centroid']:.4f}")

print("\nMEMORIZATION DIAGNOSTIC")
print(f"  reasoning KL | correct: {summary['mean_kl_reasoning_when_correct']}")
print(f"  reasoning KL | wrong  : {summary['mean_kl_reasoning_when_wrong']}")
print(f"  answer KL    | correct: {summary['mean_kl_answer_when_correct']}")
print(f"  answer KL    | wrong  : {summary['mean_kl_answer_when_wrong']}")
print(f"  reasoning H  | correct: {summary['mean_reasoning_entropy_when_correct']}")
print(f"  reasoning H  | wrong  : {summary['mean_reasoning_entropy_when_wrong']}")
print("=" * 55)

In [ ]:
# Load per-sample records from jsonl
records = []
with open(f"{OUTPUT_DIR}/samples.jsonl") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records)} sample records")

# Quick look at first record
r = records[0]
print(f"\nExample (index {r['index']}):")
print(f"  gold          : {r['gold']}")
print(f"  rollout answers: {r['answers']}")
print(f"  majority      : {r['selected_majority']}  correct={r['correct_majority']}")
print(f"  kl_centroid   : {r['selected_kl_centroid']}  correct={r['correct_kl']}")
print(f"  reasoning KL  : {r['mean_kl_reasoning']}")
print(f"  answer KL     : {r['mean_kl_answer']}")
print(f"  reasoning H   : {r['mean_reasoning_entropy']}")

In [ ]:
# Scatter: reasoning KL vs answer KL, coloured by correctness
import matplotlib.pyplot as plt
import numpy as np

correct = [r for r in records if r['correct_majority'] and r['mean_kl_reasoning'] is not None]
wrong   = [r for r in records if not r['correct_majority'] and r['mean_kl_reasoning'] is not None]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reasoning KL distribution
axes[0].hist([r['mean_kl_reasoning'] for r in correct], bins=20, alpha=0.6, label='correct', color='steelblue')
axes[0].hist([r['mean_kl_reasoning'] for r in wrong],   bins=20, alpha=0.6, label='wrong',   color='tomato')
axes[0].set_title('Reasoning region KL')
axes[0].set_xlabel('Mean pairwise KL')
axes[0].legend()

# Answer KL distribution
axes[1].hist([r['mean_kl_answer'] for r in correct], bins=20, alpha=0.6, label='correct', color='steelblue')
axes[1].hist([r['mean_kl_answer'] for r in wrong],   bins=20, alpha=0.6, label='wrong',   color='tomato')
axes[1].set_title('Answer region KL')
axes[1].set_xlabel('Mean pairwise KL')
axes[1].legend()

plt.suptitle('KL by region — correct vs wrong questions', fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/kl_by_region.png", dpi=150)
plt.show()
print(f"Saved to {OUTPUT_DIR}/kl_by_region.png")

In [ ]:
# Interpretation guide
r_kl_c = summary['mean_kl_reasoning_when_correct']
r_kl_w = summary['mean_kl_reasoning_when_wrong']
a_kl_c = summary['mean_kl_answer_when_correct']
a_kl_w = summary['mean_kl_answer_when_wrong']

print("INTERPRETATION")
if r_kl_c is not None and r_kl_w is not None:
    reasoning_gap = r_kl_w - r_kl_c
    answer_gap    = (a_kl_w or 0) - (a_kl_c or 0)
    if reasoning_gap > 0.01 and answer_gap > 0.01:
        print("  Both reasoning and answer KL separate correct from wrong.")
        print("  Consistent with genuine reasoning.")
    elif reasoning_gap < 0.005 and answer_gap > 0.01:
        print("  Only answer KL separates correct from wrong.")
        print("  Consistent with pattern-matching: answer clusters, reasoning is noise.")
    else:
        print("  Mixed signal — check the histograms above.")

## 9. Save Results Back to Drive

Results are already written into the repo folder on Drive incrementally during the run.
This cell copies them to a dedicated results folder if you want a separate archive.

In [ ]:
import shutil

ARCHIVE_DIR = "/content/drive/MyDrive/self_checking_results"
shutil.copytree(OUTPUT_DIR, ARCHIVE_DIR, dirs_exist_ok=True)
print(f"Results archived to {ARCHIVE_DIR}")
print("Files:", os.listdir(ARCHIVE_DIR))